In [2]:
import requests
import datetime
import pandas as pd

In [3]:
def get_timestamps(filter, resolution, region = "DE", ):
    response = requests.get(f"https://smard.api.proxy.bund.dev/app/chart_data/{filter}/{region}/index_{resolution}.json")
    if response.status_code == 200:
        data = response.json()
        ## TODO: Check if data is valid
    return data

In [9]:
wasser_data = get_timestamps(1226, "quarterhour", "DE")

In [4]:
relevant_filter = [1225, 1226, 1228, 4066, 4067, 4068, 4070, 410, 4359, 4387, 4169, 3791, 123, 125, 715, 5097, 122]
filter_meaning = ["wind_off", "water", "else_renew", "biomass", "wind_on", "solar", "pump", "netzlast", "residual", "pump_verbr", "preis", "prog_off", "prog_on", "prog_solar", "prog_else", "prog_wind_solar", "prog_all" ]

df_filter = pd.DataFrame({"filter": relevant_filter, "meaning": filter_meaning})

In [ ]:
wasser_data['timestamps']

In [5]:
def create_data_table(filter, resolution, timestamps_list, region = "DE"):
    comp_list = []
    for timestamp in timestamps_list:
        response = requests.get(f"https://smard.api.proxy.bund.dev/app/chart_data/{filter}/{region}/{filter}_{region}_{resolution}_{timestamp}.json")
        if response.status_code == 200:
            data = response.json()
            comp_list.extend(data["series"])

    return comp_list

In [7]:
wasser_data["timestamps"][0:1]

[1419807600000]

In [6]:
df_filter.head()

,filter,meaning
0,1225,wind_off
1,1226,water
2,1228,else_renew
3,4066,biomass
4,4067,wind_on


In [7]:
water_series = create_data_table(1226, "quarterhour", wasser_data["timestamps"][0:2], "DE")

NameError: name 'wasser_data' is not defined

In [8]:
dt_json = {}

In [9]:

for i in range(len(df_filter)):
    filter = df_filter.iloc[i,0]
    timestamps = get_timestamps(filter, "quarterhour", "DE")
    meaning = df_filter.iloc[i,1]
    dt_json[meaning] = create_data_table(filter, "quarterhour", timestamps["timestamps"][0:2], "DE" )
    

In [34]:
def check_and_clean_data(data_dict):

    # for key, pairs in dt_json.items():
    #     dt_json[key] = [pair for pair in pairs if pair[1] is not None]

    for key, pairs in dt_json.items():
        dt_json[key] = sorted(
            [pair for pair in pairs if pair[1] is not None],
            key=lambda pair: pair[0]
        )

In [32]:
check_and_clean_data(dt_json)

In [35]:
def find_timestamp_gaps(dt_json, expected_gap_ms=15 * 60 * 1000):
    """
    Checks each key in dt_json for gaps between consecutive timestamps.

    Args:
        dt_json: dict like {
            "some_key": [[timestamp_ms, value], [timestamp_ms, value], ...],
            ...
        }
        expected_gap_ms: expected distance between timestamps in milliseconds.
                         Default = 15 minutes.

    Returns:
        dict where each key contains:
            - "has_gaps": True/False
            - "gaps": list of gap details
    """
    result = {}

    for key, pairs in dt_json.items():
        gaps = []

        for i in range(1, len(pairs)):
            prev_ts = pairs[i - 1][0]
            curr_ts = pairs[i][0]
            diff = curr_ts - prev_ts

            if diff != expected_gap_ms:
                gaps.append({
                    "prev_timestamp": prev_ts,
                    "current_timestamp": curr_ts,
                    "difference_ms": diff,
                    "missing_intervals": max(0, diff // expected_gap_ms - 1)
                })

        result[key] = {
            "has_gaps": len(gaps) > 0,
            "gaps": gaps
        }

    return result

In [36]:
gap_info = find_timestamp_gaps(dt_json)

for key, info in gap_info.items():
    if info["has_gaps"]:
        print(f"{key} has gaps:")
        for gap in info["gaps"]:
            print(gap)
    else:
        print(f"{key} has no gaps")

wind_off has no gaps
water has no gaps
else_renew has no gaps
biomass has no gaps
wind_on has no gaps
solar has no gaps
pump has no gaps
netzlast has no gaps
residual has no gaps
pump_verbr has no gaps
preis has no gaps
prog_off has gaps:
{'prev_timestamp': 1420325100000, 'current_timestamp': 1420412400000, 'difference_ms': 87300000, 'missing_intervals': 96}
prog_on has no gaps
prog_solar has no gaps
prog_else has gaps:
{'prev_timestamp': 1420325100000, 'current_timestamp': 1420412400000, 'difference_ms': 87300000, 'missing_intervals': 96}
prog_wind_solar has gaps:
{'prev_timestamp': 1420325100000, 'current_timestamp': 1420412400000, 'difference_ms': 87300000, 'missing_intervals': 96}
prog_all has no gaps


In [17]:
water_series = [i for i in water_series if i[1] is not None]

In [18]:
water_series[0:5]

[[1420066800000, 288.25],
 [1420067700000, 287.75],
 [1420068600000, 292.75],
 [1420069500000, 289.5],
 [1420070400000, 295.25]]

In [12]:

df_water = pd.DataFrame({'timestamp': [i for i in water_series[0]], 'value': [i for i in water_series[1]], })

In [ ]:

data["timestamps"]
dt = datetime.datetime.fromtimestamp(data["timestamps"][2] / 1000.0)
print(dt)


2015-01-12 00:00:00


In [33]:
1629065700000
data["timestamps"]
dt = datetime.datetime.fromtimestamp(1629068400000 / 1000.0)
print(dt)

2021-08-16 01:00:00
